In [0]:
-- ============================================
-- COMPREHENSIVE EDA: ACLED Jordan Events
-- ============================================

-- 1. Dataset Overview
SELECT 
  COUNT(*) as total_events,
  COUNT(DISTINCT event_id_cnty) as unique_events,
  MIN(event_date) as earliest_date,
  MAX(event_date) as latest_date,
  COUNT(DISTINCT year) as years_covered,
  COUNT(DISTINCT admin1) as governorates_affected,
  SUM(fatalities) as total_fatalities,
  AVG(fatalities) as avg_fatalities_per_event,
  COUNT(CASE WHEN fatalities > 0 THEN 1 END) as events_with_fatalities
FROM info_env_jordan.bronze.acled_jordan_events;


SELECT disorder_type,
    event_type,
    sub_event_type,
    COUNT(*)
FROM info_env_jordan.bronze.acled_jordan_events
WHERE fatalities > 0 
GROUP BY disorder_type,
    event_type,
    sub_event_type
ORDER BY 1 ASC, 2 ASC, 3 ASC     

-- 2. Temporal Trends: Monthly event frequency
SELECT 
  DATE_TRUNC('month', TO_DATE(event_date)) as month,
  COUNT(*) as event_count,
  COUNT(DISTINCT event_type) as event_type_diversity,
  SUM(fatalities) as monthly_fatalities,
  COUNT(CASE WHEN civilian_targeting != '' THEN 1 END) as civilian_targeting_events
FROM info_env_jordan.bronze.acled_jordan_events
GROUP BY DATE_TRUNC('month', TO_DATE(event_date))
ORDER BY month;

-- 3. Event Type Distribution with Severity
SELECT 
  disorder_type,
  event_type,
  sub_event_type,
  COUNT(*) as event_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_of_total,
  SUM(fatalities) as total_fatalities,
  ROUND(AVG(fatalities), 2) as avg_fatalities
FROM info_env_jordan.bronze.acled_jordan_events
GROUP BY disorder_type, event_type, sub_event_type
ORDER BY event_count DESC;

-- 4. Geographic Distribution: Top locations
SELECT 
  admin1 as governorate,
  admin2 as district,
  location,
  COUNT(*) as event_count,
  COUNT(DISTINCT event_type) as event_types,
  SUM(fatalities) as total_fatalities,
  STRING_AGG(DISTINCT event_type, ', ') as event_types_list
FROM info_env_jordan.bronze.acled_jordan_events
WHERE admin1 IS NOT NULL
GROUP BY admin1, admin2, location
ORDER BY event_count DESC
LIMIT 30;

-- 5. Actor Analysis: Key actors and their roles
SELECT 
  actor1,
  assoc_actor_1,
  inter1 as actor_type,
  interaction,
  COUNT(*) as event_count,
  SUM(fatalities) as total_fatalities,
  COUNT(CASE WHEN civilian_targeting != '' THEN 1 END) as civilian_targeting_count,
  STRING_AGG(DISTINCT event_type, ', ') as event_types_involved
FROM info_env_jordan.bronze.acled_jordan_events
GROUP BY actor1, assoc_actor_1, inter1, interaction
ORDER BY event_count DESC
LIMIT 30;

-- 6. Violence & Civilian Impact
SELECT 
  CASE 
    WHEN fatalities = 0 THEN 'No fatalities'
    WHEN fatalities BETWEEN 1 AND 5 THEN '1-5 fatalities'
    WHEN fatalities BETWEEN 6 AND 10 THEN '6-10 fatalities'
    ELSE '10+ fatalities'
  END as fatality_range,
  COUNT(*) as event_count,
  SUM(fatalities) as total_fatalities,
  STRING_AGG(DISTINCT event_type, ', ') as event_types
FROM info_env_jordan.bronze.acled_jordan_events
GROUP BY fatality_range
ORDER BY 
  CASE fatality_range
    WHEN 'No fatalities' THEN 1
    WHEN '1-5 fatalities' THEN 2
    WHEN '6-10 fatalities' THEN 3
    ELSE 4
  END;

-- 7. Interaction Patterns: Actor vs Actor dynamics
SELECT 
  interaction,
  inter1 as actor1_type,
  inter2 as actor2_type,
  COUNT(*) as event_count,
  SUM(fatalities) as total_fatalities,
  COUNT(CASE WHEN civilian_targeting != '' THEN 1 END) as civilian_targeted,
  STRING_AGG(DISTINCT event_type, ', ') as event_types
FROM info_env_jordan.bronze.acled_jordan_events
WHERE interaction IS NOT NULL AND interaction != ''
GROUP BY interaction, inter1, inter2
ORDER BY event_count DESC
LIMIT 20;

-- 8. Data Quality Check
SELECT 
  'event_id_cnty' as field,
  COUNT(*) as total_records,
  COUNT(event_id_cnty) as non_null,
  COUNT(*) - COUNT(event_id_cnty) as null_count,
  ROUND((COUNT(event_id_cnty) * 100.0 / COUNT(*)), 2) as completeness_pct
FROM info_env_jordan.bronze.acled_jordan_events
UNION ALL
SELECT 'event_date', COUNT(*), COUNT(event_date), COUNT(*) - COUNT(event_date), ROUND((COUNT(event_date) * 100.0 / COUNT(*)), 2)
FROM info_env_jordan.bronze.acled_jordan_events
UNION ALL
SELECT 'admin1', COUNT(*), COUNT(admin1), COUNT(*) - COUNT(admin1), ROUND((COUNT(admin1) * 100.0 / COUNT(*)), 2)
FROM info_env_jordan.bronze.acled_jordan_events
UNION ALL
SELECT 'location', COUNT(*), COUNT(location), COUNT(*) - COUNT(location), ROUND((COUNT(location) * 100.0 / COUNT(*)), 2)
FROM info_env_jordan.bronze.acled_jordan_events
UNION ALL
SELECT 'latitude', COUNT(*), COUNT(latitude), COUNT(*) - COUNT(latitude), ROUND((COUNT(latitude) * 100.0 / COUNT(*)), 2)
FROM info_env_jordan.bronze.acled_jordan_events
UNION ALL
SELECT 'actor1', COUNT(*), COUNT(actor1), COUNT(*) - COUNT(actor1), ROUND((COUNT(actor1) * 100.0 / COUNT(*)), 2)
FROM info_env_jordan.bronze.acled_jordan_events;